# 05 — Change Impact Simulation

This notebook quantifies the **expected rebuild cost** of changes flowing through the dependency graph.
We combine git-history change probabilities with transitive rebuild costs to compute a *pain score*
for every target, then run Monte Carlo simulations to estimate daily build overhead under the current
and proposed (merge / split) graph topologies.

Key outputs:
- Per-target pain score ranking
- Merge and split what-if simulations
- Monte Carlo distribution of daily rebuild cost
- Sensitivity analysis across different git-history windows

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics, critical_path, node_centrality
from build_optimiser.simulation import rebuild_cost, expected_daily_cost, simulate_merge, simulate_split, monte_carlo_rebuild_cost
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

In [ ]:
# ── Change probability model from git history ─────────────────────────
#
# P(target changes on a given day) ≈ target_commits / total_commits
# We normalise over the observation window so probabilities sum to 1.

commit_col = "git_commit_count_total"
total_commits = target_metrics[commit_col].sum()

change_prob = target_metrics[["cmake_target", commit_col]].copy()
change_prob["change_probability"] = change_prob[commit_col] / total_commits
change_prob = change_prob.sort_values("change_probability", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CDF of change probability
sorted_probs = np.sort(change_prob["change_probability"].values)[::-1]
cumulative = np.cumsum(sorted_probs)
axes[0].plot(range(1, len(cumulative) + 1), cumulative, lw=2)
axes[0].axhline(0.8, ls="--", color="red", alpha=0.6, label="80 % of change mass")
axes[0].set_xlabel("Number of targets (ranked by change frequency)")
axes[0].set_ylabel("Cumulative probability")
axes[0].set_title("CDF of target-change probability")
axes[0].legend()

# Top 20 most frequently changed targets
top20 = change_prob.head(20)
axes[1].barh(top20["cmake_target"], top20["change_probability"], color="steelblue")
axes[1].invert_yaxis()
axes[1].set_xlabel("Change probability")
axes[1].set_title("Top-20 most frequently changed targets")

plt.tight_layout()
plt.show()

n_80 = int(np.searchsorted(cumulative, 0.8)) + 1
print(f"{n_80} targets account for 80 % of all changes ({n_80 / len(change_prob) * 100:.1f} % of targets)")

In [ ]:
# ── Expected rebuild cost per target (pain score ranking) ─────────────
#
# pain_score(t) = P(change_t) × rebuild_cost(t)
# where rebuild_cost includes the target itself plus all transitive dependants.

pain_records = []
for _, row in target_metrics.iterrows():
    t = row["cmake_target"]
    if t not in G:
        continue
    rc = rebuild_cost(G, t, target_metrics)
    ec = expected_daily_cost(G, t, target_metrics, target_metrics)
    n_dependants = len(nx.ancestors(G, t))
    pain_records.append({
        "cmake_target": t,
        "rebuild_cost_ms": rc,
        "expected_daily_cost_ms": ec,
        "change_probability": row[commit_col] / total_commits if total_commits > 0 else 0,
        "n_dependants": n_dependants,
    })

pain_df = pd.DataFrame(pain_records).sort_values("expected_daily_cost_ms", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 6))
top30 = pain_df.head(30)
colors = plt.cm.YlOrRd(top30["change_probability"] / top30["change_probability"].max())
bars = ax.barh(top30["cmake_target"], top30["expected_daily_cost_ms"] / 1000, color=colors)
ax.invert_yaxis()
ax.set_xlabel("Expected daily rebuild cost (seconds)")
ax.set_title("Pain Score Ranking — Top 30 Targets")
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(0, top30["change_probability"].max()))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Change probability")
plt.tight_layout()
plt.show()

print("Top 10 pain scores:")
display(pain_df.head(10))

In [ ]:
# ── Merge simulation ──────────────────────────────────────────────────
#
# Identify candidate pairs for merging: targets that always rebuild together
# (high Jaccard similarity in their dependant sets) and are small individually.

# Find pairs of targets that share many dependants
targets_in_graph = [t for t in target_metrics["cmake_target"] if t in G]
dependant_sets = {t: nx.ancestors(G, t) for t in targets_in_graph}

merge_candidates = []
for i, t1 in enumerate(targets_in_graph):
    for t2 in targets_in_graph[i + 1:]:
        s1, s2 = dependant_sets[t1], dependant_sets[t2]
        union_size = len(s1 | s2)
        if union_size == 0:
            continue
        jaccard = len(s1 & s2) / union_size
        if jaccard > 0.6:
            merge_candidates.append({"target_a": t1, "target_b": t2, "jaccard": jaccard})

merge_cand_df = pd.DataFrame(merge_candidates).sort_values("jaccard", ascending=False)
print(f"Found {len(merge_cand_df)} merge-candidate pairs (Jaccard > 0.6)")
display(merge_cand_df.head(10))

# Simulate top-3 merge candidates
merge_results = []
for _, row in merge_cand_df.head(3).iterrows():
    result = simulate_merge(G, [row["target_a"], row["target_b"]], target_metrics)
    merge_results.append({
        "targets": f"{row['target_a']} + {row['target_b']}",
        "before_cost_ms": result["before_total_cost"],
        "after_cost_ms": result["after_cost"],
        "delta_ms": result["delta"],
        "delta_pct": result["delta"] / max(result["before_total_cost"], 1) * 100,
    })

merge_res_df = pd.DataFrame(merge_results)
print("\nMerge simulation results:")
display(merge_res_df)

In [ ]:
# ── Split simulation ──────────────────────────────────────────────────
#
# Large, highly-central targets are split candidates. We simulate splitting
# each into two roughly equal halves by file count.

centrality = node_centrality(G)
split_scores = []
for t in targets_in_graph:
    t_row = target_metrics[target_metrics["cmake_target"] == t]
    if t_row.empty:
        continue
    n_files = int(t_row["file_count"].iloc[0]) if "file_count" in t_row.columns else 0
    bc = centrality.get(t, 0)
    compile_time = int(t_row["compile_time_sum_ms"].iloc[0]) if "compile_time_sum_ms" in t_row.columns else 0
    split_scores.append({"cmake_target": t, "centrality": bc, "file_count": n_files, "compile_time_ms": compile_time})

split_df = pd.DataFrame(split_scores)
# Normalise and combine to rank
for col in ["centrality", "file_count", "compile_time_ms"]:
    vmax = split_df[col].max()
    split_df[f"{col}_norm"] = split_df[col] / vmax if vmax > 0 else 0

split_df["split_score"] = (
    split_df["centrality_norm"] * 0.4
    + split_df["file_count_norm"] * 0.3
    + split_df["compile_time_ms_norm"] * 0.3
)
split_df = split_df.sort_values("split_score", ascending=False).reset_index(drop=True)

print("Top split candidates:")
display(split_df[["cmake_target", "centrality", "file_count", "compile_time_ms", "split_score"]].head(10))

# Simulate split on top candidate
top_split_target = split_df.iloc[0]["cmake_target"]
target_files = file_metrics[file_metrics["cmake_target"] == top_split_target]["source_file"].tolist()

if len(target_files) >= 2:
    mid = len(target_files) // 2
    group_a, group_b = target_files[:mid], target_files[mid:]
    split_result = simulate_split(G, top_split_target, [group_a, group_b], target_metrics, file_metrics)
    print(f"\nSplit simulation for '{top_split_target}':")
    print(f"  Before cost : {split_result['before_cost']:>12,} ms")
    print(f"  After cost  : {split_result['after_total_cost']:>12,} ms")
    print(f"  Delta       : {split_result['delta']:>+12,} ms  ({split_result['delta'] / max(split_result['before_cost'], 1) * 100:+.1f} %)")
else:
    print(f"Target '{top_split_target}' has too few files to split.")

In [ ]:
# ── Monte Carlo simulation (sample random workdays) ───────────────────
#
# Each simulation represents one workday: for every target we flip a biased
# coin (P = change_probability) and compute the total rebuild cost of all
# affected targets.

mc = monte_carlo_rebuild_cost(G, target_metrics, n_simulations=5000, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of daily rebuild cost
axes[0].hist(mc["raw"] / 1000, bins=60, edgecolor="white", alpha=0.8, color="steelblue")
axes[0].axvline(mc["mean"] / 1000, color="red", ls="--", lw=2, label=f"Mean: {mc['mean']/1000:.0f}s")
axes[0].axvline(mc["p95"] / 1000, color="orange", ls="--", lw=2, label=f"P95: {mc['p95']/1000:.0f}s")
axes[0].set_xlabel("Daily rebuild cost (seconds)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Monte Carlo – Daily Rebuild Cost Distribution")
axes[0].legend()

# Box plot
axes[1].boxplot(mc["raw"] / 1000, vert=True, widths=0.5)
axes[1].set_ylabel("Daily rebuild cost (seconds)")
axes[1].set_title("Box Plot")

plt.tight_layout()
plt.show()

print(f"Mean daily rebuild cost : {mc['mean']/1000:>10,.1f} s")
print(f"Median                  : {mc['median']/1000:>10,.1f} s")
print(f"Std dev                 : {mc['std']/1000:>10,.1f} s")
print(f"95th percentile         : {mc['p95']/1000:>10,.1f} s")

In [ ]:
# ── Sensitivity analysis (3, 6, 12 month windows) ─────────────────────
#
# Re-weight change probabilities using different look-back windows to see
# how stable the pain ranking is.

# Construct windowed commit counts if columns are available
window_cols = {
    "3 months": "git_commit_count_3m",
    "6 months": "git_commit_count_6m",
    "12 months": "git_commit_count_total",
}

# Fall back: if windowed columns don't exist, synthesise them by scaling
available_windows = {}
for label, col in window_cols.items():
    if col in target_metrics.columns:
        available_windows[label] = col

if len(available_windows) < 3:
    # Synthesise approximate windows from total by applying decay factors
    base_col = commit_col
    for label, factor in [("3 months", 0.25), ("6 months", 0.5), ("12 months", 1.0)]:
        synth_col = f"_synth_{label.replace(' ', '_')}"
        target_metrics[synth_col] = (target_metrics[base_col] * factor).astype(int).clip(lower=0)
        # Add Poisson noise to make it realistic
        rng = np.random.default_rng(hash(label) % 2**31)
        noise = rng.poisson(lam=1, size=len(target_metrics))
        target_metrics[synth_col] = (target_metrics[synth_col] + noise).clip(lower=0)
        available_windows[label] = synth_col

mc_results = {}
for label, col in sorted(available_windows.items()):
    # Temporarily swap commit column for MC simulation
    tmp = target_metrics.copy()
    tmp["git_commit_count_total"] = tmp[col]
    mc_res = monte_carlo_rebuild_cost(G, tmp, n_simulations=3000, seed=42)
    mc_results[label] = mc_res

fig, ax = plt.subplots(figsize=(10, 5))
positions = []
labels = []
data = []
for i, (label, res) in enumerate(sorted(mc_results.items())):
    positions.append(i + 1)
    labels.append(label)
    data.append(res["raw"] / 1000)

bp = ax.boxplot(data, positions=positions, widths=0.5, patch_artist=True)
for patch, color in zip(bp["boxes"], ["#a1c9f4", "#ffb482", "#8de5a1"]):
    patch.set_facecolor(color)
ax.set_xticklabels(labels)
ax.set_ylabel("Daily rebuild cost (seconds)")
ax.set_title("Sensitivity Analysis – Monte Carlo by Git-History Window")
plt.tight_layout()
plt.show()

summary_rows = []
for label, res in sorted(mc_results.items()):
    summary_rows.append({
        "window": label,
        "mean_s": res["mean"] / 1000,
        "median_s": res["median"] / 1000,
        "p95_s": res["p95"] / 1000,
        "std_s": res["std"] / 1000,
    })
display(pd.DataFrame(summary_rows))